In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [2]:
from datasets import load_dataset
import pandas as pd


## Loading the MGSD dataset.

dataset = load_dataset("wu981526092/MGSD")

data = dataset['train']
df = data.to_pandas()


## Loading the MentalManip dataset

dataset_2 = load_dataset("audreyeleven/MentalManip", "mentalmanip_maj")
data_2 = dataset_2["train"]
df_2 = data_2.to_pandas()

Some datasets params were ignored: ['license']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


In [4]:
## Balanced dataset from MGSD

sample_sizes = {
    'stereotype': 40,
    'anti-stereotype': 40,
    'unrelated': 20,
}

balanced_samples = []

for label, size in sample_sizes.items():
    subset = df[df["label"] == label].sample(n=size, random_state=42)
    balanced_samples.append(subset)

sample_mgsd = pd.concat(balanced_samples).sample(frac=1, random_state=42).reset_index(drop=True)
print(sample_mgsd['label'].value_counts())


## Balanced dataset from MentalManip

sample_sizes_mentalmanip = [60, 40]
balanced_samples_manip = []

for idx, size in enumerate(sample_sizes_mentalmanip):
    subset = df_2[df_2["manipulative"] == idx].sample(n=size, random_state=42)
    balanced_samples_manip.append(subset)

sample_mentalmanip = pd.concat(balanced_samples_manip).sample(frac=1, random_state=42).reset_index(drop=True)
print(sample_mentalmanip["manipulative"].value_counts())

label
anti-stereotype    40
stereotype         40
unrelated          20
Name: count, dtype: int64
manipulative
0    60
1    40
Name: count, dtype: int64


In [ ]:
from dotenv import load_dotenv
from tree_of_thought import TreeOfThought
import openai
import os, torch, numpy as np
from utils import call_llm
from plot_tree import plot_tree
import json
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

load_dotenv()

ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        raise ValueError(f"Missing {name} API key: `{var}` must be set in the environment.")


client = openai.OpenAI(api_key= os.getenv("API_KEY_OPENAI"))
model = "gpt-4o-mini"

## Zero-shot prompting

In [ ]:
import pandas as pd
from tqdm import tqdm
import os, json
from stereotype_def import stereotype_definition, stereotype_definition_short
from mentalmanip_def import mental_manipulation_def1_short, mental_manipulation_def1
from case_studies import stereotypes, manipulation
from zero_shot import ZeroShot
from sklearn.metrics import classification_report, confusion_matrix


case_name = "stereotype" # "stereotype" "manipulation"

if case_name.lower() == "manipulation":
    case = manipulation
    task_definition = mental_manipulation_def1
    data = sample_mentalmanip
    text_col = "dialogue"
    label_col = "manipulative"
    output_file = "results/low/results_mentalmanip_zero_shot_prompt_long.csv"

elif case_name.lower() == "stereotype":
    case = stereotypes
    task_definition = stereotype_definition
    data = sample_mgsd
    text_col = "text_no_marker"
    label_col = "label"
    output_file = "results/low/results_stereotype_zero_shot_prompt_long.csv"

else:
    raise ValueError(f"Unknown case name: {case_name}")


zero_shot_classifier = ZeroShot(
    case=case,
    client=client,
    model=model,
    max_tokens=100,
    task_definition=task_definition
)


rows = []


for idx, row in tqdm(data.iterrows(), total=len(data)):
    text = row[text_col]
    true_label = row[label_col]
    
    if isinstance(true_label, str):
        true_label = true_label.strip()

    predicted_label = zero_shot_classifier.classify(text)
    mapped_label = case["label_map"].get(predicted_label.strip(), list(case["label_map"].values())[-1])

    results = {
        "sample_id": idx,
        "text": text,
        "true_label": true_label,
        "pred_label": mapped_label,
        "max_tokens": zero_shot_classifier.max_tokens,
        "tokens_used": zero_shot_classifier.total_tokens,
        "prompt_tokens": zero_shot_classifier.total_prompt_tokens,
        "completion_tokens": zero_shot_classifier.total_completion_tokens,
        "latency": zero_shot_classifier.total_latency,
    }

    rows.append(results)

df_out = pd.DataFrame(rows)
df_out.to_csv(output_file, index=False)
print(f"=== Saved {len(df_out)} rows to {output_file}")

if case_name == "manipulation":
    y_true = df_out["true_label"].astype(int)
    y_pred = df_out["pred_label"].astype(int)

elif case_name == "stereotype":
    y_true = df_out["true_label"]
    y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()


print("=== Classification Report ===\n")
print(classification_report(y_true, y_pred))


print("\n=== Confusion Matrix ===\n")
labels = sorted(set(y_true) | set(y_pred))
conf_matrix = confusion_matrix(y_true, y_pred)
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

accuracy = (y_true == y_pred).mean()
print(f"\n=== Accuracy: {accuracy:.2%} ===")

## Few-shots prompting